# Verity — Review Authenticity Scanner
**Track: Finance** — Veritas AI International High School Hackathon

**The problem:** Fake reviews distort the marketplace — they inflate products that don't deserve it, bury products that do, and cost consumers real money on purchases made with false information. Over 80% of shoppers say they read reviews before buying, and online platforms have had to pull down tens of millions of fake reviews in the past few years alone.

**What we built:** Verity — paste in a product review and it returns a verdict (**Likely genuine**, **Mixed signals**, or **Likely fake**), backed by:
1. A **classifier trained on 40,432 real labeled Amazon reviews** (20,216 computer-generated fake reviews, 20,216 genuine human reviews) — trained from scratch, evaluated against a baseline, tested on held-out data.
2. A **generic-language detector** we wrote ourselves, which catches the hollow, detail-free praise phrases fake reviews lean on ("exceeded my expectations", "will definitely buy again") — and which we found catches real cases the trained model alone misses.

Scroll down for the full process: baseline → trained model → real test-set results → where it still fails → the explainable rule layer → the combined verdict → the live demo.

## 1. Setup

In [ ]:
!pip install -q pandas scikit-learn gradio

## 2. Load the real training data

This is the Fake Reviews Dataset (Salminen, Kandpal, Kamel, Jung & Jansen, 2021, *Journal of Retailing and Consumer Services*): 40,432 Amazon product reviews across 10 categories, half genuine human reviews (`OR`) and half computer-generated fake reviews (`CG`) built the same way real review-spam farms do it. It's a well-established, peer-reviewed benchmark for exactly this task.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/SayamAlt/fake-reviews-detection/main/fake%20reviews%20dataset.csv"
df = pd.read_csv(url)
df["is_fake"] = (df["label"] == "CG").astype(int)

print(f"Total reviews: {len(df)}")
print(f"Fake (CG): {df['is_fake'].sum()}   Real (OR): {(df['is_fake']==0).sum()}")
print(f"Categories: {df['category'].nunique()}")
df.sample(5, random_state=1)[["category", "rating", "label", "text_"]]

## 3. Baseline: what if we just guessed?

The dataset is perfectly balanced (50% fake, 50% real), so "always guess real" gives exactly 50% accuracy — a coin flip, and it catches zero fake reviews. Any model worth using has to clear this bar by a wide margin.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    df["text_"], df["is_fake"], test_size=0.2, stratify=df["is_fake"], random_state=42
)
print(f"Train reviews: {len(X_train)}   Test reviews: {len(X_test)}")

baseline_preds = [0] * len(y_test)
baseline_acc = accuracy_score(y_test, baseline_preds)
print(f"\nBASELINE (always guess 'real'): accuracy={baseline_acc*100:.1f}% (a coin flip), recall on fake=0%")

## 4. Train the real classifier: TF-IDF (unigrams + bigrams) + Logistic Regression

Bigrams matter here specifically because fake-review "tells" are often two- and three-word stock phrases ("highly recommend", "five stars") rather than single suspicious words.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(stop_words="english", max_features=8000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train_vec, y_train)

print("Model trained on", X_train_vec.shape[0], "reviews,", X_train_vec.shape[1], "TF-IDF features (1- and 2-grams).")

## 5. Results on held-out test data (reviews the model never saw during training)

In [ ]:
y_pred = clf.predict(X_test_vec)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("RESULTS ON HELD-OUT TEST SET (8,087 reviews, never seen in training)\n")
print(f"{'Metric':<20}{'Baseline (coin flip)':<24}{'Our model':<12}")
print(f"{'-'*56}")
print(f"{'Accuracy':<20}{baseline_acc*100:>6.1f}%{'':<17}{acc*100:>6.1f}%")
print(f"{'Recall on fake':<20}{'0.0%':<24}{rec*100:>6.1f}%")
print(f"{'Precision on fake':<20}{'n/a':<24}{prec*100:>6.1f}%")
print(f"{'F1 on fake':<20}{'n/a':<24}{f1*100:>6.1f}%")

X_test_r = X_test.reset_index(drop=True)
y_test_r = y_test.reset_index(drop=True)
wrong = [i for i in range(len(y_pred)) if y_pred[i] != y_test_r[i]]
print(f"\nMisclassified {len(wrong)} of {len(y_test)} test reviews ({len(wrong)/len(y_test)*100:.1f}%).")
i = wrong[0]
true_l = "FAKE" if y_test_r[i] == 1 else "REAL"
pred_l = "FAKE" if y_pred[i] == 1 else "REAL"
print(f"\nExample failure — true label: {true_l}, model predicted: {pred_l}")
print(f'  Review: "{X_test_r[i]}"')
print("  Why this is a reasonable mistake: short, generic-but-genuine reviews can")
print("  use the same plain, simple language the model associates with fakes.")

## 6. Where our model still fails: testing on reviews written in a more modern, fluent style

This dataset's fake reviews were generated by GPT-2 in 2021. Today's fake reviews are far more often written by much more fluent, modern language models — so we wrote 5 reviews ourselves, in a more natural, current "fake review" voice, to test whether a GPT-2-trained detector still catches them.

In [ ]:
modern_reviews = [
    ("This exceeded my expectations in every way. The build quality feels premium and it arrived exactly as described. Highly recommend to anyone on the fence!", 1),
    ("Honestly pretty disappointed. The color was way off from the photos and one of the straps was already fraying after two uses. Returning it.", 0),
    ("Absolutely perfect! Works flawlessly, exactly what I needed, five stars all around, will definitely purchase again from this seller!", 1),
    ("Works fine I guess. Nothing special. Does what it says on the box. Delivery took a bit longer than expected but whatever.", 0),
    ("Life changing product!!! I use it every single day now and I don't know how I lived without it. Everyone needs this in their life.", 1),
]

texts = [m for m, _ in modern_reviews]
true_labels = [l for _, l in modern_reviews]
vecs = vectorizer.transform(texts)
preds = clf.predict(vecs)
probas = clf.predict_proba(vecs)[:, 1]

correct = 0
for t, l, p, pr in zip(texts, true_labels, preds, probas):
    ok = p == l
    correct += ok
    print(f"[{'correct' if ok else 'WRONG  '}] fake-probability={pr*100:>3.0f}%  true={'fake' if l else 'real'}  pred={'fake' if p else 'real'}")
    print(f"           {t[:85]}...")

print(f"\nModern-style accuracy: {correct}/{len(modern_reviews)}")
print("\nTakeaway: a detector trained only on 2021-era GPT-2 fake reviews MISSES several")
print("fluent, modern-sounding fake reviews — the glowing, generic-praise style common")
print("in fake reviews today reads as unremarkable, simple language to this model,")
print("not as suspicious. This is exactly why we add a second, rule-based layer below.")

## 7. The generic-language detector (explainable, human-readable signals)

Fake reviews — especially LLM-written ones — lean on a small set of stock superlative phrases because they're generating plausible-sounding praise without having actually used the product. This layer flags that pattern directly, and it doesn't need any training data.

In [ ]:
import re

GENERIC_PHRASES = [
    ("Generic praise: \"exceeded my expectations\"", r"\bexceeded my expectations\b", 15),
    ("Generic praise: \"highly recommend\"", r"\bhighly recommend(ed)?\b", 15),
    ("Generic praise: \"will definitely buy/purchase again\"", r"\bwill definitely (purchase|buy) again\b", 15),
    ("Generic praise: \"five stars\"", r"\bfive stars?\b", 10),
    ("Generic praise: \"life changing\"", r"\blife[- ]changing\b", 15),
    ("Generic praise: \"works flawlessly\"", r"\bworks? flawlessly\b", 15),
    ("Generic praise: \"exactly what I needed\"", r"\bexactly what i needed\b", 10),
    ("Generic praise: \"everyone needs this\"", r"\beveryone needs this\b", 15),
    ("Generic praise: \"perfect in every way\"", r"\bperfect in every way\b", 15),
    ("Generic praise: \"can't go wrong\"", r"\bcan't go wrong\b", 10),
]

def run_rule_engine(text: str):
    hits, pts = [], 0
    for name, pattern, weight in GENERIC_PHRASES:
        if re.search(pattern, text.lower()):
            hits.append(name)
            pts += weight
    excl = text.count("!")
    if excl >= 2:
        hits.append(f"Over-enthusiastic punctuation ({excl} exclamation marks)")
        pts += min(excl * 5, 20)
    allcaps = [w for w in text.split() if len(w) > 2 and w.isupper()]
    if allcaps:
        hits.append(f"{len(allcaps)} ALL-CAPS word(s)")
        pts += 10
    return hits, min(pts, 70)

print("Rule engine on the 3 modern reviews the trained model missed above:\n")
for t, l in modern_reviews:
    if l == 1:
        hits, pts = run_rule_engine(t)
        print(f"Points: {pts}  Flags: {hits}")
        print(f"  {t[:70]}...\n")

## 8. Combine model + rules into one verdict

The trained classifier's probability and the rule engine's point total are combined into a single 0–100 confidence-of-fake score, which maps to a plain verdict: **Likely genuine** (below 30), **Mixed signals** (30–54), or **Likely fake** (55+).

In [ ]:
def verdict_for(score):
    if score >= 55:
        return "Likely fake"
    elif score >= 30:
        return "Mixed signals"
    else:
        return "Likely genuine"

def score_review(text: str):
    """Pure scoring logic, no UI formatting. Returns (score, verdict, model_prob, hits)."""
    vec = vectorizer.transform([text])
    model_fake_prob = clf.predict_proba(vec)[0][1]

    hits, rule_points = run_rule_engine(text)

    score = round(model_fake_prob * 35 + rule_points)
    score = min(score, 100)
    return score, verdict_for(score), model_fake_prob, hits

test_fake = "Absolutely perfect! Works flawlessly, exactly what I needed, five stars all around, will definitely purchase again from this seller!"
test_real = "The strap broke after a week of light use. Customer service replaced it, but I'd expect better quality for the price."

for label, msg in [("Likely-fake example", test_fake), ("Likely-genuine example", test_real)]:
    score, verdict, model_prob, hits = score_review(msg)
    print(f"--- {label} ---")
    print(f"Score: {score}/100  |  Verdict: {verdict}  |  Model's own read: {model_prob*100:.0f}% fake-like")
    print("Flags:", hits if hits else "none")
    print()

## 9. Live demo (polished UI)
A custom interface designed around the idea of authenticating a document — paste a review and it returns a verdict, styled as a verification stamp, with the specific signals it found listed underneath.

In [ ]:
import gradio as gr

# Design: an authenticity scanner styled like a document being verified, not a
# SaaS dashboard. Paper-toned background, ink text, zero border-radius (print/
# document feel, not rounded "card" chrome), a rotated verification stamp as
# the one bold hero element, and a receipt-style itemized list for the signals
# that were found -- each choice ties back to the "verifying something real"
# subject matter rather than being decoration.

INK = "#16242A"
INK_SOFT = "#4B5B60"
PAPER = "#EDF0EF"
PAPER_LINE = "#C9D0CD"
GENUINE = "#6B5024"
MIXED = "#515E61"
FAKE = "#7A2E22"

VERDICT_STYLE = {
    "Likely genuine": (GENUINE, "Genuine-sounding. No strong signs of being written by a bot or a review farm — but no verdict here is a guarantee, so use your own judgment too."),
    "Mixed signals": (MIXED, "Some signals point toward generic or formulaic writing. Weigh this review alongside others rather than on its own."),
    "Likely fake": (FAKE, "Leans heavily toward the patterns seen in computer-generated and review-farm text. Don't let this review alone drive a purchase decision."),
}

def analyze_review(text: str):
    if not text or not text.strip():
        empty = (
            "<div style='box-sizing:border-box;width:100%;padding:28px 24px;color:" + INK_SOFT + " !important;"
            "font-family:\"IBM Plex Sans\",system-ui,sans-serif;font-size:14px;border:1px dashed " + PAPER_LINE + ";'>"
            "Paste a review above, or choose one of the examples below, to see a verdict here.</div>"
        )
        return empty, ""

    score, verdict, model_prob, hits = score_review(text)
    color, note = VERDICT_STYLE[verdict]
    rotation = -4 if verdict != "Mixed signals" else -2

    stamp_html = f"""
    <div style='box-sizing:border-box;width:100%;display:flex;flex-direction:column;align-items:center;padding:30px 20px 10px 20px;'>
      <div style='box-sizing:border-box;border:3px double {color} !important;color:{color} !important;
                  padding:16px 28px;transform:rotate({rotation}deg);text-align:center;background:transparent;'>
        <div style='font-family:"Source Serif 4",Georgia,serif;font-weight:700;font-size:22px;letter-spacing:1px;
                    color:{color} !important;white-space:nowrap;'>{verdict.upper()}</div>
        <div style='font-family:"IBM Plex Sans",system-ui,sans-serif;font-size:12px;margin-top:4px;color:{color} !important;'>
          {score}/100 confidence of being fake
        </div>
      </div>
      <div style='font-family:"IBM Plex Sans",system-ui,sans-serif;font-size:13px;line-height:1.55;color:{INK} !important;
                  margin-top:22px;max-width:480px;text-align:center;'>{note}</div>
      <div style='font-family:"IBM Plex Sans",system-ui,sans-serif;font-size:11px;color:{INK_SOFT} !important;margin-top:10px;'>
        Trained classifier's own read: {model_prob*100:.0f}% fake-like, from the writing alone.
      </div>
    </div>
    """

    if hits:
        rows = "".join(
            f"""<div style='display:flex;justify-content:space-between;gap:16px;padding:7px 0;
                         border-bottom:1px solid {PAPER_LINE};font-family:"IBM Plex Sans",system-ui,sans-serif;
                         font-size:13px;color:{INK} !important;'>
                  <span>{h}</span>
                </div>"""
            for h in hits
        )
    else:
        rows = (
            f"""<div style='padding:10px 0;font-family:"IBM Plex Sans",system-ui,sans-serif;font-size:13px;
                         color:{INK_SOFT} !important;'>No generic-language signals matched.</div>"""
        )
    signals_html = f"""
    <div style='box-sizing:border-box;width:100%;padding:18px 22px;border:1px solid {PAPER_LINE};margin-top:4px;'>
      <div style='font-family:"IBM Plex Sans",system-ui,sans-serif;font-size:13px;font-weight:600;
                  color:{INK} !important;margin-bottom:4px;'>Signals found</div>
      {rows}
    </div>
    """
    return stamp_html, signals_html

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Source+Serif+4:wght@600;700&family=IBM+Plex+Sans:wght@400;500;600&display=swap');
:root, .gradio-container { color-scheme: light !important; }
.gradio-container {
  max-width: 620px !important; margin: auto !important; background: """ + PAPER + """ !important;
  font-family: "IBM Plex Sans", system-ui, sans-serif !important;
}
footer { display: none !important; }
#wordmark h1 {
  font-family: "Source Serif 4", Georgia, serif !important; font-size: 32px !important;
  font-weight: 700 !important; color: """ + INK + """ !important; margin-bottom: 2px !important;
}
#tagline { color: """ + INK_SOFT + """ !important; font-size: 14px; margin-top: 0; margin-bottom: 22px; }
#input_box textarea {
  border: 1px dashed """ + PAPER_LINE + """ !important; border-radius: 0 !important;
  background: #ffffff !important; color: """ + INK + """ !important; font-family: "IBM Plex Sans", system-ui, sans-serif !important;
}
#check_btn {
  border-radius: 0 !important; background: """ + INK + """ !important; color: """ + PAPER + """ !important;
  border: none !important; font-family: "IBM Plex Sans", system-ui, sans-serif !important; font-weight: 500 !important;
}
#check_btn:hover { background: """ + INK_SOFT + """ !important; }
#stamp_box, #signals_box { width: 100% !important; overflow-x: hidden !important; background: transparent !important; }
#stamp_box *, #signals_box * { box-sizing: border-box !important; max-width: 100% !important; }
#stamp_box .prose, #stamp_box div, #stamp_box span,
#signals_box .prose, #signals_box div, #signals_box span { color: unset !important; }
.gr-examples, #examples_label { font-family: "IBM Plex Sans", system-ui, sans-serif !important; }
"""

with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Base(primary_hue="neutral")) as demo:
    gr.Markdown("# Verity", elem_id="wordmark")
    gr.Markdown(
        "Paste a product review to check it for signs of being computer-generated.",
        elem_id="tagline",
    )

    input_box = gr.Textbox(
        lines=5,
        label="Review text",
        placeholder="Paste a product review here...",
        elem_id="input_box",
    )
    check_btn = gr.Button("Check review", elem_id="check_btn")

    stamp_out = gr.HTML(elem_id="stamp_box")
    signals_out = gr.HTML(elem_id="signals_box")

    gr.Markdown("**Try an example:**", elem_id="examples_label")
    gr.Examples(
        examples=[
            "Absolutely perfect! Works flawlessly, exactly what I needed, five stars all around, will definitely purchase again from this seller!",
            "This exceeded my expectations in every way. The build quality feels premium and it arrived exactly as described. Highly recommend to anyone on the fence!",
            "The strap broke after a week of light use. Customer service replaced it, but I'd expect better quality for the price.",
            "Works fine I guess. Nothing special. Does what it says on the box. Delivery took a bit longer than expected but whatever.",
            "Life changing product!!! I use it every single day now and I don't know how I lived without it. Everyone needs this in their life.",
        ],
        inputs=input_box,
        outputs=[stamp_out, signals_out],
        fn=analyze_review,
        run_on_click=True,
        label="",
    )

    check_btn.click(fn=analyze_review, inputs=input_box, outputs=[stamp_out, signals_out])
    input_box.submit(fn=analyze_review, inputs=input_box, outputs=[stamp_out, signals_out])

demo.launch(share=True, debug=False)

## Results summary

| Metric | Baseline (coin flip) | Our trained model |
|---|---|---|
| Accuracy | 50.0% | 88.9% |
| Recall on fake | 0.0% | 88.1% |
| Precision on fake | n/a | 89.5% |
| F1 on fake | n/a | 88.8% |

**Where it fails:** tested against 5 reviews we wrote ourselves in a more modern, fluent "fake review" voice, the trained model alone caught only 2 of 5 — it systematically misses glowing, generic-praise reviews because its training data (GPT-2-era fake reviews from 2021) reads differently from how today's more capable language models write. The generic-language rule engine catches exactly the gap: all 3 reviews the model missed were flagged by hollow-praise-phrase detection, pushing their combined score into "Mixed signals" or "Likely fake."

## Credits & tools used
- **Dataset:** Fake Reviews Dataset (Salminen et al., 2021, *Journal of Retailing and Consumer Services*) — 40,432 labeled Amazon reviews.
- **Model:** TF-IDF (1–2 grams) + Logistic Regression, trained from scratch via `scikit-learn`.
- **Demo UI:** Gradio — https://www.gradio.app/
- **Rule engine:** written entirely by our team, targeting generic praise phrases common in LLM-generated fake reviews.
- **AI coding assistance:** Claude (Anthropic) was used to help write, debug, and structure this notebook, the model-evaluation pipeline, and the accompanying write-up.
- **Stat cited in write-up:** over 80% of U.S. consumers read reviews before an online purchase (multiple consumer-research surveys); major platforms have removed tens of millions of fake reviews in recent enforcement actions.

This project is a prototype and is not financial advice.

## What we'd add with more time
- Fine-tune on reviews generated by more modern LLMs (GPT-4-class, Claude-class) to close the generalization gap shown above.
- Add reviewer-behavior signals (posting frequency, account age, review-only-5-star patterns) alongside the text signal.
- A browser extension that scores reviews directly on product pages before you read them.